# AdPilot Pro – Final Master AI Pipeline Simulation Notebook

This notebook represents the complete simulation of the future production system loading all individual agent checkpoints.


## 1. Executive Summary & Business Goals



### Pipeline Architecture:
This master pipeline integrates all trained agents (Strategy, Research, Content, Design, Analytics, Optimizer, CV, Embeddings, RAG) to ingest a raw Campaign Brief and output an optimized advertising strategy, creative copy, visual assets, and budget plan.



## 2. Load Model Checkpoints


In [ ]:
import joblib
import torch
import os
import sys
import numpy as np

# Load all exported weights
models_dir = "research/models"

# 1. Strategy Model
strategy_model = joblib.load("models/strategy/strategy_model.pkl")

# 2. Research Model
research_model = joblib.load(os.path.join(models_dir, "research", "research_model.pkl"))

# 3. Content Model
content_model = joblib.load(os.path.join(models_dir, "content", "content_model.pkl"))

# 4. Design Model
design_model = joblib.load(os.path.join(models_dir, "design", "creative_quality_score.pkl"))

# 5. Analytics Model
analytics_model = joblib.load(os.path.join(models_dir, "analytics", "analytics_model.pkl"))

# 6. Optimizer Model
# Bind PolicyNetwork class to main so pickling finds it
import __main__
class PolicyNetwork(torch.nn.Module):
    def __init__(self, state_dim, action_dim):
        super(PolicyNetwork, self).__init__()
        self.fc = torch.nn.Sequential(
            torch.nn.Linear(state_dim, 64),
            torch.nn.ReLU(),
            torch.nn.Linear(64, action_dim),
            torch.nn.Tanh()
        )
    def forward(self, x):
        return self.fc(x)
__main__.PolicyNetwork = PolicyNetwork

optimizer_model = joblib.load(os.path.join(models_dir, "optimizer", "optimizer_model.pkl"))

# 7. CV Model
cv_model = joblib.load(os.path.join(models_dir, "cv", "cv_model.pkl"))

# 8. Embeddings Model
embedding_model = joblib.load(os.path.join(models_dir, "embeddings", "embedding_model.pkl"))

# 9. RAG Model
rag_model = joblib.load(os.path.join(models_dir, "rag", "rag_model.pkl"))

print("All individual agent models loaded successfully.")


## 3. Campaign Generation Simulation


In [ ]:
# Ingest Campaign Brief
brief = {
    "topic": "Luxury real estate property listings near the lake.",
    "target_audience": "High-net-worth investors.",
    "initial_budget": 5000.0,
    "duration": 30
}

# 1. Strategy Prediction (Propensity check)
# expects 11 features: [age, balance, duration, campaign, pdays, previous, housing_enc, loan_enc, age_group_enc, job_enc, marital_enc]
strat_input = np.array([[45, 10000.0, 300, 2, -1, 1, 1, 0, 1, 1, 1]])
strat_pred = strategy_model.predict(strat_input)

# 2. Content Quality Score
# Mock TF-IDF features matrix for Ridge
content_input = np.random.randn(1, 50)
content_score = content_model.predict(content_input)

# 3. Analytics ROAS prediction
# [age, balance, duration, campaign, previous, bal_dur_ratio, campaign_efficiency]
analytics_input = np.array([[45, 10000.0, 300, 2, 1, 10000.0/301.0, 300*2]])
roas_pred = analytics_model.predict(analytics_input)

print("Pipeline Simulation Outputs:")
print(f"- Strategy Conversion Propensity: {strat_pred[0]}")
print(f"- Creative Copy Quality Score: {content_score[0]:.4f}")
print(f"- Expected Campaign ROAS: {roas_pred[0]:.4f}")


## 4. Pipeline Latency Breakdown


In [ ]:
import time
import pandas as pd

# Measure step-by-step latency
breakdown = []

for step, action in [
    ("Strategy", lambda: strategy_model.predict(strat_input)),
    ("Content", lambda: content_model.predict(content_input)),
    ("Analytics", lambda: analytics_model.predict(analytics_input))
]:
    start = time.perf_counter()
    action()
    latency = (time.perf_counter() - start) * 1000.0
    breakdown.append({"Step": step, "Latency (ms)": latency})

breakdown_df = pd.DataFrame(breakdown)
print("Pipeline Execution Timeline Breakdown:")
print(breakdown_df)


## 5. Export Pipeline Diagnostics


In [ ]:
import os
import json
from datetime import datetime

out_dir = "research/models/pipeline"
os.makedirs(out_dir, exist_ok=True)

# Export master config
master_config = {
    "strategy_version": "1.0.0",
    "research_version": "1.0.0",
    "content_version": "1.0.0",
    "pipeline_execution_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}
with open(os.path.join(out_dir, "pipeline_config.json"), "w") as f:
    json.dump(master_config, f, indent=2)

print("Master pipeline metadata successfully saved.")
